In [2]:
import pandas as pd
from pathlib import Path

# Read the balanced dataset (no header expected; last column is the label)
input_path = Path('dataset_tratado_balanceado.csv')
df = pd.read_csv(input_path, header=None)

# Normalize columns: 9 board positions + 1 label
expected_cols = 10
if df.shape[1] < expected_cols:
    raise ValueError(f'Expected at least {expected_cols} columns, got {df.shape[1]}')
if df.shape[1] > expected_cols:
    df = df.iloc[:, :expected_cols]  # Keep first 10 if extra
df.columns = [f'pos{i}' for i in range(9)] + ['label']

# Stratified per-class split: keep class proportions in train/test
train_parts = []
test_parts = []
random_state = 42

for label, group in df.groupby('label'):
    n = len(group)
    # Target 10% in test (rounded), with at least 1 when class has > 1 sample
    n_test = int(round(n * 0.1))
    if n_test < 1 and n > 1:
        n_test = 1
    if n_test >= n:
        n_test = max(0, n - 1)

    if n_test == 0:
        train_parts.append(group)
    else:
        g_test = group.sample(n=n_test, random_state=random_state)
        g_train = group.drop(g_test.index)
        test_parts.append(g_test)
        train_parts.append(g_train)

# Concatenate and shuffle
df_train = pd.concat(train_parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
df_test = pd.concat(test_parts).sample(frac=1, random_state=random_state).reset_index(drop=True)

# Print class distributions to verify balance
print('Original class counts:')
print(df['label'].value_counts())
print('\nTrain class counts:')
print(df_train['label'].value_counts())
print('\nTest class counts:')
print(df_test['label'].value_counts())

# Save without header and index to match existing dataset format
df_train.to_csv('dataset_treino.csv', header=False, index=False)
df_test.to_csv('dataset_testes.csv', header=False, index=False)
print(f"Saved dataset_treino.csv ({len(df_train)} rows) and dataset_testes.csv ({len(df_test)} rows)")

Original class counts:
label
Fim_X_Vence          200
Fim_O_Vence          200
Possibilidade_fim    200
Tem jogo             200
Empate               184
Name: count, dtype: int64

Train class counts:
label
Fim_X_Vence          180
Possibilidade_fim    180
Tem jogo             180
Fim_O_Vence          180
Empate               166
Name: count, dtype: int64

Test class counts:
label
Possibilidade_fim    20
Fim_X_Vence          20
Tem jogo             20
Fim_O_Vence          20
Empate               18
Name: count, dtype: int64
Saved dataset_treino.csv (886 rows) and dataset_testes.csv (98 rows)
